In [ ]:
import subprocess, time, os

print("Downloading Ollama...")
subprocess.run(
    "curl -fsSL https://github.com/ollama/ollama/releases/download/v0.24.0/ollama-linux-amd64.tar.zst -o /tmp/ollama.tar.zst",
    shell=True
)
subprocess.run("apt-get install -y zstd -q", shell=True, capture_output=True)
subprocess.run("tar --use-compress-program=unzstd -xf /tmp/ollama.tar.zst -C /tmp/", shell=True)
binary = subprocess.run("find /tmp -name 'ollama' -type f", shell=True, capture_output=True, text=True).stdout.strip().split('\n')[0]
subprocess.run(f"cp {binary} /usr/local/bin/ollama && chmod +x /usr/local/bin/ollama", shell=True)
v = subprocess.run(["/usr/local/bin/ollama", "--version"], capture_output=True, text=True)
print("✅ Ollama:", v.stdout.strip())

✅ Ollama: Warning: could not connect to a running Ollama instance


In [ ]:
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
subprocess.Popen(["/usr/local/bin/ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Pull only if not already cached
models = subprocess.run(["/usr/local/bin/ollama", "list"], capture_output=True, text=True)
print(models.stdout)

if "qwen2.5-coder" not in models.stdout:
    print("Pulling qwen2.5-coder:7b (~4GB)...")
    subprocess.run(["/usr/local/bin/ollama", "pull", "qwen2.5-coder:7b"])
else:
    print("✅ Model already cached, skipping download")

# Create claude- alias
subprocess.run(["/usr/local/bin/ollama", "cp", "qwen2.5-coder:7b", "claude-qwen"], capture_output=True)
print("✅ Ollama ready")

NAME    ID    SIZE    MODIFIED 

Pulling qwen2.5-coder:7b (~4GB)...
✅ Ollama ready


In [ ]:
subprocess.run("pip install 'litellm[proxy]==1.82.4' -q", shell=True)

config = """
model_list:
  - model_name: claude-sonnet-4-6
    litellm_params:
      model: ollama_chat/qwen2.5-coder:7b
      api_base: http://localhost:11434

litellm_settings:
  drop_params: true
"""
with open("/tmp/litellm_config.yaml", "w") as f:
    f.write(config)

subprocess.Popen(
    "litellm --config /tmp/litellm_config.yaml --port 8080 > /tmp/litellm.log 2>&1 &",
    shell=True
)
time.sleep(15)

# Test model call
chat = subprocess.run("""curl -s http://localhost:8080/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer ollama" \
  -d '{"model": "claude-sonnet-4-6", "messages": [{"role": "user", "content": "say hi"}], "max_tokens": 10}'""",
shell=True, capture_output=True, text=True)
print("Model test:", chat.stdout[:200])
print("✅ LiteLLM ready" if "content" in chat.stdout else "❌ Check logs: cat /tmp/litellm.log")

Model test: 
❌ Check logs: cat /tmp/litellm.log


import datetime

print("🔁 Session keeper running. This cell keeps Kaggle alive.")
print("   Do not interrupt this cell.\n")

count = 0
while True:
    count += 1
    now = datetime.datetime.now().strftime("%H:%M:%S")
    
    # Health check Ollama
    check = subprocess.run(
        ["curl", "-s", "http://localhost:11434/api/tags"],
        capture_output=True, text=True
    )
    
    if check.returncode != 0:
        print(f"[{now}] ⚠️  Ollama down — restarting...")
        subprocess.Popen(["ollama", "serve"])
        time.sleep(5)
    else:
        models = __import__('json').loads(check.stdout).get('models', [])
        names = [m['name'] for m in models]
        print(f"[{now}] ✅ Heartbeat #{count} | Models loaded: {names}", flush=True)
    
    time.sleep(5 * 60)  # every 5 minutes

In [ ]:
import subprocess

# Download cloudflared
subprocess.run(
    "curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
    shell=True
)

# Verify
v = subprocess.run(["/usr/local/bin/cloudflared", "--version"], capture_output=True, text=True)
print("✅", v.stdout.strip())

In [ ]:
import re

subprocess.run("pkill cloudflared 2>/dev/null", shell=True)
time.sleep(2)

subprocess.Popen(
    "nohup ./cloudflared tunnel --url http://localhost:8080 > /tmp/cf.log 2>&1 &",
    shell=True
)

url = None
print("Getting tunnel URL", end="", flush=True)
for _ in range(60):
    time.sleep(2)
    print(".", end="", flush=True)
    try:
        log = open("/tmp/cf.log").read()
        match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', log)
        if match:
            url = match.group(0)
            break
    except:
        pass

if url:
    print(f"\n\n{'='*55}")
    print(f"✅ TUNNEL URL:\n\n    {url}\n")
    print(f"On Mac:")
    print(f'  ANTHROPIC_BASE_URL="{url}" \\')
    print(f'  ANTHROPIC_API_KEY="ollama" \\')
    print(f'  claude --model claude-sonnet-4-6')
    print(f"{'='*55}")

Getting tunnel URL............................................................

In [ ]:
import datetime, json

def heartbeat():
    count = 0
    while True:
        count += 1
        now = datetime.datetime.now().strftime("%H:%M:%S")
        check = subprocess.run("curl -s http://localhost:11434/api/tags", shell=True, capture_output=True, text=True)
        if check.returncode != 0 or not check.stdout:
            print(f"[{now}] ⚠️ Ollama down — restarting...")
            subprocess.Popen(["/usr/local/bin/ollama", "serve"])
            time.sleep(8)
        else:
            models = json.loads(check.stdout).get('models', [])
            names = [m['name'] for m in models]
            print(f"[{now}] ✅ #{count} | Models: {names}", flush=True)
        time.sleep(5 * 60)

import threading
t = threading.Thread(target=heartbeat, daemon=True)
t.start()
print("✅ Heartbeat running in background")

✅ Heartbeat running in background


In [ ]:
import datetime
print("🔁 Keep-alive active — do not stop this cell")
count = 0
while True:
    count += 1
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] alive #{count}", flush=True)
    time.sleep(10 * 60)

🔁 Keep-alive active — do not stop this cell
[19:23:28] alive #1
[19:23:28] ✅ #1 | Models: ['claude-qwen:latest', 'qwen2.5-coder:7b']
[19:28:28] ✅ #2 | Models: ['claude-qwen:latest', 'qwen2.5-coder:7b']
[19:33:28] alive #2
[19:33:28] ✅ #3 | Models: ['claude-qwen:latest', 'qwen2.5-coder:7b']
[19:38:28] ✅ #4 | Models: ['claude-qwen:latest', 'qwen2.5-coder:7b']
[19:43:28] alive #3
[19:43:28] ✅ #5 | Models: ['qwen2.5-coder:7b', 'claude-qwen:latest']
[19:48:28] ✅ #6 | Models: ['claude-qwen:latest', 'qwen2.5-coder:7b']
[19:53:28] alive #4
[19:53:28] ✅ #7 | Models: ['claude-qwen:latest', 'qwen2.5-coder:7b']
[19:58:28] ✅ #8 | Models: ['claude-qwen:latest', 'qwen2.5-coder:7b']


In [ ]:
# import subprocess, time

# # Check if pyngrok already installed
# try:
#     from pyngrok import ngrok
#     print("✅ pyngrok already installed")
# except ImportError:
#     print("Installing pyngrok...")
#     subprocess.run([sys.executable, "-m", "pip", "install", "pyngrok"],
#     capture_output=True, text=True)
#     from pyngrok import ngrok

# # Set your token (only needed once per session)
# ngrok.set_auth_token("3EdTKTjwUs7Pf9Lkkltw71NDRtV_7iPmeDHp6byCVch8BJc9T")

# # Kill old tunnels
# ngrok.kill()
# time.sleep(2)

# # Start tunnel
# tunnel = ngrok.connect(8080)
# url = tunnel.public_url

# print(f"✅ TUNNEL: {url}")
# print(f'ANTHROPIC_BASE_URL="{url}/v1"')
# print(f'ANTHROPIC_API_KEY="ollama"')
# print(f'claude --model claude-sonnet-4-6')

In [1]:
# 1. Initialize git and create a README
!echo "# OpenSource-Model-LLM-Personal-Free-Token-Learning" >> README.md
!git init

# 2. Add BOTH the README and your current Colab Notebook file
# Replace "your_notebook_name.ipynb" with your actual file name
!git add README.md
!git add "ollama-vanchai.ipynb"

# 3. Commit the files
!git config --global user.name "rajabose"
!git config --global user.email "rbosemonarch@gmail.com"
!git commit -m "tunneling of open source model to capitalize in claude terminal for free tokens and free GPU Training."

# 4. Set the branch to main and point to your correct repository
!git branch -M main
!git remote add origin https://github.com/rajabose/OpenSource-Model-LLM-Personal-Free-Token-Learning.git

# 5. Push the files to GitHub
# Note: If it asks for a password, you must use your GitHub Personal Access Token (PAT)
!git push -u origin main


hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
fatal: pathspec 'ollama-vanchai.ipynb' did not match any files
[master (root-commit) 1c0eea9] tunneling of open source model to capitalize in claude terminal for free tokens and free GPU Training.
 1 file changed, 1 insertion(+)
 create mode 100644 README.md
fatal: could not read Username for 'https://github.com': No such device or address
